In [65]:
import regex as re
from collections import defaultdict

In [66]:
corpus  = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""

special_tokens = ["<|endoftext|>"]

In [67]:
vocab = {i: bytes([i]) for i in range(256)} # 0..255
vocab_idx = 256

In [68]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
# pretokenize
def pre_tokenize(corpus: str) -> list[str]:
    return re.findall(PAT, corpus)

In [69]:
pre_tokens = pre_tokenize(corpus)

In [70]:
pre_tokens

['low',
 ' low',
 ' low',
 ' low',
 ' low',
 '\n',
 'lower',
 ' lower',
 ' widest',
 ' widest',
 ' widest',
 '\n',
 'newest',
 ' newest',
 ' newest',
 ' newest',
 ' newest',
 ' newest']

In [71]:
tuple(pre_tokens[0].encode('utf-8'))

(108, 111, 119)

In [72]:
# count
def get_pre_token_freq(pre_tokens):
    d = defaultdict(int)

    for t in pre_tokens:
        d[tuple(t.encode('utf-8'))] += 1

    return d

d = get_pre_token_freq(pre_tokens)

In [73]:
def get_byte_pair_freq(d):
    byte_pair_freq = defaultdict(int)

    for tok_bytes, freq in d.items():
        for i in range(len(tok_bytes)-1):
            byte_pair_freq[(tok_bytes[i], tok_bytes[i+1])] += freq

    return byte_pair_freq

byte_pair_freq = get_byte_pair_freq(d)

In [74]:
best_pair = max(byte_pair_freq, key=byte_pair_freq.get)
best_pair

(101, 115)

In [75]:
merges = []

In [76]:
def merge(d, best_pair, vocab_idx):
    new_d = defaultdict(int)

    for tok_bytes, freq in d.items():
        new_bytes = []
        i = 0
        while i < len(tok_bytes):
            if i != len(tok_bytes)-1 and (tok_bytes[i], tok_bytes[i+1]) == best_pair:
                new_bytes.append(vocab_idx)
                i+=2
            else:
                new_bytes.append(tok_bytes[i])
                i+=1

        new_d[tuple(new_bytes)] += freq
    return new_d

In [77]:
merge(d, best_pair, vocab_idx)
merges.append(best_pair)

In [78]:
# vocab[vocab_idx] = best_pair
vocab[vocab_idx] = vocab[best_pair[0]] + vocab[best_pair[1]]
vocab_idx += 1